# Lab 00 — Admin Setup

One-time RBAC setup for the GenAI Learning labs.  
Run this notebook as a user with **ACCOUNTADMIN** access.  

Follows [Snowflake RBAC best practices](https://docs.snowflake.com/en/user-guide/security-access-control-considerations):  
- **USERADMIN** creates roles  
- **SYSADMIN** creates warehouses and databases  
- **SECURITYADMIN** manages grants  
- **ACCOUNTADMIN** grants account-level privileges (`CREATE DATABASE`)  
- Custom roles are granted to **SYSADMIN** to maintain the standard role hierarchy

## Step 1 — Create the Role

Use USERADMIN to create the dedicated lab role.

In [ ]:
USE ROLE USERADMIN;

CREATE ROLE IF NOT EXISTS DS_ROLE
    COMMENT = 'Hands-on role for GenAI Learning labs';

## Step 2 — Create the Warehouse

Use SYSADMIN to create the compute warehouse. XSmall is sufficient for all labs.

In [ ]:
USE ROLE SYSADMIN;

CREATE WAREHOUSE IF NOT EXISTS DS_WH
    WAREHOUSE_SIZE = 'XSMALL'
    AUTO_SUSPEND  = 60
    AUTO_RESUME   = TRUE;

## Step 3 — Role Hierarchy and Grants

Use SECURITYADMIN to place DS_ROLE under SYSADMIN and grant object-level privileges.

In [ ]:
USE ROLE SECURITYADMIN;

-- Place DS_ROLE in the standard role hierarchy under SYSADMIN
GRANT ROLE DS_ROLE TO ROLE SYSADMIN;

-- Warehouse access
GRANT USAGE ON WAREHOUSE DS_WH TO ROLE DS_ROLE;

-- Sample data used by several labs (TPC-H)
GRANT IMPORTED PRIVILEGES ON DATABASE SNOWFLAKE_SAMPLE_DATA TO ROLE DS_ROLE;

-- Cortex AI functions (required for all labs)
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE DS_ROLE;

## Step 4 — Account-Level Privileges

`CREATE DATABASE` is an account-level privilege that requires **ACCOUNTADMIN** to grant.

In [ ]:
USE ROLE ACCOUNTADMIN;

GRANT CREATE DATABASE ON ACCOUNT TO ROLE DS_ROLE;

## Step 5 — Governance Labs (Optional)

Labs 16–18 query `SNOWFLAKE.ACCOUNT_USAGE` views. Skip this cell if you don't plan to run the governance section.

In [ ]:
USE ROLE SECURITYADMIN;

GRANT DATABASE ROLE SNOWFLAKE.USAGE_VIEWER TO ROLE DS_ROLE;

## Step 6 — Assign the Role to Your User

Replace `<your_username>` with the Snowflake username that will run the labs.

In [ ]:
USE ROLE SECURITYADMIN;

GRANT ROLE DS_ROLE TO USER <your_username>;

## Verify Setup

Switch to DS_ROLE and confirm warehouse access.

In [ ]:
USE ROLE DS_ROLE;
USE WAREHOUSE DS_WH;

SELECT
    CURRENT_ROLE()      AS role,
    CURRENT_WAREHOUSE() AS warehouse,
    CURRENT_USER()      AS user;

Confirm the role can create the lab database.

In [ ]:
CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;

SELECT CURRENT_DATABASE() AS database, CURRENT_SCHEMA() AS schema;

## Next Steps

Setup complete. Proceed to **Lab 01 — COMPLETE Basics** (`01-foundations/01-complete-basics/`).  

All 18 labs use `USE ROLE DS_ROLE` and `USE WAREHOUSE DS_WH` in their first cell.

## Teardown

To remove all lab objects and the role/warehouse when you're done:

In [ ]:
USE ROLE DS_ROLE;
DROP DATABASE IF EXISTS GENAI_LEARNING;

USE ROLE SYSADMIN;
DROP WAREHOUSE IF EXISTS DS_WH;

USE ROLE USERADMIN;
DROP ROLE IF EXISTS DS_ROLE;